In [6]:
!mkdir -p ~/.kaggle

In [7]:
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [8]:
!kaggle competitions download -c birdclef-2026

100% 15.0G/15.0G [01:36<00:00, 166MB/s]



In [9]:
!mkdir dataset
!unzip -q birdclef-2026.zip -d dataset

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import IPython.display as ipd
import numpy as np

sns.set_theme(style="whitegrid")

train_df = pd.read_csv('dataset/train.csv')
taxonomy_df = pd.read_csv('dataset/taxonomy.csv')

try:
    soundscapes_df = pd.read_csv('dataset/train_soundscapes_labels.csv')
    has_soundscapes = True
except FileNotFoundError:
    has_soundscapes = False
    print("Файл train_soundscapes_labels.csv не знайдено, перевір назву.")

print(f"Загальна кількість записів у train.csv: {len(train_df)}")
print(f"Унікальних класів у train.csv: {train_df['primary_label'].nunique()}")
print(f"Кількість класів, які вимагає змагання (з taxonomy): {len(taxonomy_df)}")

if has_soundscapes:
    train_classes = set(train_df['primary_label'].unique())
    soundscape_classes = set()
    for labels in soundscapes_df['primary_label'].dropna():
        soundscape_classes.update(labels.split(';'))

    missing_in_train = soundscape_classes - train_classes
    print(f"Кількість класів, які є ТІЛЬКИ в soundscapes: {len(missing_in_train)}")
    print(f"Ці класи: {missing_in_train}")

Загальна кількість записів у train.csv: 35549
Унікальних класів у train.csv: 206
Кількість класів, які вимагає змагання (з taxonomy): 234
Кількість класів, які є ТІЛЬКИ в soundscapes: 28
Ці класи: {'47158son17', '47158son08', '47158son11', '47158son21', '47158son05', '47158son12', '47158son22', '47158son24', '47158son20', '47158son09', '517063', '47158son01', '1491113', '47158son04', '25073', '47158son14', '47158son15', '47158son23', '47158son18', '47158son25', '47158son19', '47158son13', '47158son16', '47158son10', '47158son02', '47158son03', '47158son06', '47158son07'}


In [ ]:
import os

In [9]:
from sklearn.model_selection import StratifiedKFold

train_df = train_df.reset_index(drop=True)

train_df['fold'] = -1

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['primary_label'])):
    train_df.loc[val_idx, 'fold'] = fold

print(train_df['fold'].value_counts())

train_df.to_csv('dataset/train_folds.csv', index=False)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


fold
2    7110
0    7110
3    7110
1    7110
4    7109
Name: count, dtype: int64


In [10]:
import torch
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import LabelEncoder

import pandas as pd
from sklearn.preprocessing import LabelEncoder


taxonomy_df = pd.read_csv('dataset/taxonomy.csv')
le = LabelEncoder()
le.fit(taxonomy_df['primary_label'])

train_df['label_id'] = le.transform(train_df['primary_label'])

num_classes = len(le.classes_)
print(f"Кількість унікальних класів для моделі: {num_classes}")

class BirdCLEFDataset(Dataset):
    def __init__(self, df, audio_dir, target_sr=32000, duration=5.0):
        self.df = df
        self.audio_dir = audio_dir
        self.target_sr = target_sr
        self.num_samples = int(target_sr * duration)

        self.mel_spectrogram = T.MelSpectrogram(
            sample_rate=target_sr,
            n_fft=1024,
            hop_length=512,
            n_mels=128
        )
        self.amplitude_to_db = T.AmplitudeToDB()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        audio_path = f"{self.audio_dir}/{row['filename']}"

        try:
            waveform, sr = torchaudio.load(audio_path)
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)
        except Exception as e:
            waveform = torch.zeros(1, self.num_samples)

        if waveform.shape[1] > self.num_samples:
            start = np.random.randint(0, waveform.shape[1] - self.num_samples)
            waveform = waveform[:, start:start + self.num_samples]
        elif waveform.shape[1] < self.num_samples:
            pad_length = self.num_samples - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_length))

        mel = self.mel_spectrogram(waveform)
        mel_db = self.amplitude_to_db(mel)

        # min-max normalization
        mel_min = mel_db.min()
        mel_max = mel_db.max()
        if mel_max - mel_min > 0:
            mel_normalized = (mel_db - mel_min) / (mel_max - mel_min)
        else:
            mel_normalized = mel_db - mel_min

        mel_normalized = mel_normalized.repeat(3, 1, 1)

        label = row['label_id']

        return mel_normalized, torch.tensor(label, dtype=torch.long)


Кількість унікальних класів для моделі: 234


In [11]:
test_dataset = BirdCLEFDataset(df=train_df, audio_dir='dataset/train_audio')
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=True)

for inputs, labels in test_loader:
    print(f"Розмірність спектрограм (Batch, Channels, Mels, Time): {inputs.shape}")
    print(f"Значення лейблів: {labels}")
    print(f"Максимальне значення пікселя: {inputs.max().item()}")
    print(f"Мінімальне значення пікселя: {inputs.min().item()}")
    break

Розмірність спектрограм (Batch, Channels, Mels, Time): torch.Size([4, 3, 128, 313])
Значення лейблів: tensor([110,  88, 143, 174])
Максимальне значення пікселя: 1.0
Мінімальне значення пікселя: 0.0


In [12]:
import torchvision.models as models
import torch.nn as nn

class BirdCLEFResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Поточний пристрій: {device}")

model = BirdCLEFResNet(num_classes=len(le.classes_)).to(device)

Поточний пристрій: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 197MB/s]


# BirdCLEF 2025 — Top Solutions Analysis

## Architectures

All solutions convert audio → mel spectrogram → CNN. Two main types:

**Plain CNN** — backbone + linear output layer (20th place, `TimmCNN`)

**SED model** — backbone + attention head (`AttBlockV2`) that weights each time frame. 1st place also adds GeM frequency pooling. Used by 1st and 9th place.

| Solution | Backbone(s) |
|---|---|
| 1st | `tf_efficientnet_b4/b3/b0`, `regnety_016`, `eca_nfnet_l0` |
| 2nd | `tf_efficientnetv2_s`, `eca_nfnet_l0` |
| 9th | `seresnext26t_32x4d` × 3 seeds |
| 20th | `tf_efficientnet_b0_ns` |

## Preprocessing

Standard across all solutions: 32kHz, 5s clips, mel spectrogram, 3-channel image input.

| Setting | Range |
|---|---|
| n_fft | 2048–4096 |
| n_mels | 128–384 |
| Image size | 128×256 to 384×384 |
| Normalization | per-sample (not global) |

## Augmentations

| Level | Technique | Who used it |
|---|---|---|
| Waveform | Mixup | Everyone |
| Waveform | Volume scaling | 2nd place |
| Waveform | Human voice removal (silero-vad) | 20th place |
| Spectrogram | SpecAugment (freq + time masking) | 20th, 1st |
| Spectrogram | CoarseDropout | 2nd place |
| Label | Secondary labels, label smoothing | Most solutions |

## Training

- **Loss:** FocalLoss + BCE (20th, 2nd) / BCE (9th) / CE (1st)
- **Optimizer:** AdamW + Cosine LR with warmup
- **Model Soup** (20th): average weights of best checkpoints instead of picking one
- **Pseudo-labeling** (1st, 2nd): 3–4 rounds of self-training on unlabeled soundscapes

## Inference

- 5s non-overlapping windows
- Ensemble of 3–7 diverse models
- ONNX/OpenVINO for fast CPU inference
- Temporal smoothing across neighboring windows


# BirdCLEF 2026 — Official Metric

## Macro-averaged ROC-AUC

The official metric is **Macro-averaged ROC-AUC** — replaces padded cmAP from 2025.

**ROC-AUC** asks: if you pick one window where bird X is present and one where it's absent, did your model score the positive higher? AUC = 1.0 is perfect, AUC = 0.5 is random.

**Key property:** metric is threshold-free — only ranking matters, not exact probabilities. No need to tune a threshold like "predict bird if prob > 0.5".

**Macro** = AUC computed per species, then averaged. Every species counts equally — a rare bird with 3 occurrences matters as much as a common one with 300. Species with zero test positives are skipped.

## vs Padded cmAP (2025)

cmAP (Average Precision) penalizes false positives at the top of the ranked list more than AUC does. A model can have high AUC but lower cmAP if it's overconfident on wrong species.


In [14]:
!pip install -q timm audiomentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 25.3 MB/s eta 0:00:00


In [15]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [16]:
CFG = {
    'sample_rate'    : 32000,
    'duration'       : 5,
    'n_mels'         : 128,
    'n_fft'          : 1024,
    'hop_length'     : 512,
    'img_size'       : (128, 256),
    'batch_size'     : 32,
    'num_workers'    : 2,
    'lr'             : 3e-4,
    'epochs'         : 7,
    'val_fold'       : 0,       # same validation split as Participant 1
    'model_name'     : 'efficientnet_b0',
    'seed'           : 42,
    'label_smoothing': 0.05,
    'mixup_alpha'    : 0.4,
    'mixup_prob'     : 0.5,
}

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

In [17]:
train_df    = pd.read_csv('dataset/train_folds.csv')
taxonomy_df = pd.read_csv('dataset/taxonomy.csv')

le = LabelEncoder()
le.fit(taxonomy_df['primary_label'])
train_df['label_id'] = le.transform(train_df['primary_label'])

num_classes = len(le.classes_)
print(f"Number of classes: {num_classes}")
print(f"Train samples: {(train_df['fold'] != CFG['val_fold']).sum()}, "
      f"Val samples: {(train_df['fold'] == CFG['val_fold']).sum()}")

Number of classes: 234
Train samples: 28439, Val samples: 7110


In [18]:
class BirdDataset(Dataset):
    def __init__(self, df, audio_dir, is_train=False):
        self.df          = df.reset_index(drop=True)
        self.audio_dir   = audio_dir
        self.is_train    = is_train
        self.num_samples = CFG['sample_rate'] * CFG['duration']

        self.mel = T.MelSpectrogram(
            sample_rate=CFG['sample_rate'],
            n_fft=CFG['n_fft'],
            hop_length=CFG['hop_length'],
            n_mels=CFG['n_mels'],
            f_min=50,
            f_max=14000,
        )
        self.db        = T.AmplitudeToDB(top_db=80)
        self.freq_mask = T.FrequencyMasking(freq_mask_param=15)
        self.time_mask = T.TimeMasking(time_mask_param=30)
        self.resize    = torch.nn.Upsample(
            size=CFG['img_size'], mode='bilinear', align_corners=False
        )

    def load_wave(self, path):
        try:
            wave, _ = torchaudio.load(path)
            if wave.shape[0] > 1:
                wave = wave.mean(dim=0, keepdim=True)
            if wave.shape[1] >= self.num_samples:
                start = np.random.randint(0, wave.shape[1] - self.num_samples + 1) \
                        if self.is_train else 0
                wave = wave[:, start:start + self.num_samples]
            else:
                wave = torch.nn.functional.pad(
                    wave, (0, self.num_samples - wave.shape[1])
                )
        except Exception:
            wave = torch.zeros(1, self.num_samples)
        return wave

    def wave_to_spec(self, wave):
        mel = self.mel(wave)
        mel = self.db(mel)
        mel = (mel + 80) / 80       # normalize to [0, 1]
        return mel

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = f"{self.audio_dir}/{row['filename']}"
        wave = self.load_wave(path)

        # Gaussian noise augmentation
        if self.is_train and np.random.rand() < 0.3:
            wave = wave + 0.005 * torch.randn_like(wave)

        spec = self.wave_to_spec(wave)      # (1, n_mels, T)

        # SpecAugment
        if self.is_train:
            spec = self.freq_mask(spec)
            spec = self.time_mask(spec)

        spec = self.resize(spec.unsqueeze(0)).squeeze(0)    # (1, H, W)
        spec = spec.repeat(3, 1, 1)                         # (3, H, W)

        return spec, torch.tensor(row['label_id'], dtype=torch.long)

In [19]:
def mixup_batch(x, y, alpha=CFG['mixup_alpha']):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [20]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing=CFG['label_smoothing']):
        super().__init__()
        self.gamma = gamma
        self.ce    = nn.CrossEntropyLoss(
            label_smoothing=label_smoothing, reduction='none'
        )

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        pt      = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

In [21]:
class BirdModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = timm.create_model(
            CFG['model_name'],
            pretrained=True,
            num_classes=num_classes,
            in_chans=3,
            drop_rate=0.3,
        )

    def forward(self, x):
        return self.backbone(x)

model       = BirdModel(num_classes).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Trainable parameters: 4,307,302


In [22]:
train_data = train_df[train_df['fold'] != CFG['val_fold']].reset_index(drop=True)
val_data   = train_df[train_df['fold'] == CFG['val_fold']].reset_index(drop=True)

train_ds = BirdDataset(train_data, 'dataset/train_audio', is_train=True)
val_ds   = BirdDataset(val_data,   'dataset/train_audio', is_train=False)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                          shuffle=True,  num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train batches: 889 | Val batches: 223


In [23]:
criterion = FocalLoss()
optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=1e-6
)

In [24]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    for x, y in tqdm(loader, desc='Train', leave=False):
        x, y = x.to(device), y.to(device)

        if np.random.rand() < CFG['mixup_prob']:
            x, y_a, y_b, lam = mixup_batch(x, y)
            loss = mixup_loss(criterion, model(x), y_a, y_b, lam)
        else:
            loss = criterion(model(x), y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def validate(model, loader):
    model.eval()
    total_loss = 0
    all_probs, all_labels = [], []

    with torch.no_grad():
        for x, y in tqdm(loader, desc='Val  ', leave=False):
            x, y = x.to(device), y.to(device)
            out  = model(x)
            total_loss += criterion(out, y).item()
            all_probs.append(torch.softmax(out, dim=1).cpu().numpy())
            all_labels.append(y.cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    y_onehot = np.zeros_like(all_probs)
    y_onehot[np.arange(len(all_labels)), all_labels] = 1

    present = y_onehot.sum(0) > 0
    auc = roc_auc_score(
        y_onehot[:, present], all_probs[:, present], average='macro'
    )
    return total_loss / len(loader), auc

In [25]:
best_auc = 0
history  = []

for epoch in range(CFG['epochs']):
    train_loss          = train_one_epoch(model, train_loader)
    val_loss, val_auc   = validate(model, val_loader)
    scheduler.step()

    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_auc': val_auc,
    })

    print(f"Epoch {epoch+1:02d}/{CFG['epochs']} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val AUC: {val_auc:.4f} | "
          f"LR: {optimizer.param_groups[0]['lr']:.2e}")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), 'best_efficientnet_b0.pth')
        print(f"  -> Saved! Best AUC: {best_auc:.4f}")

print(f"\nBest Val AUC: {best_auc:.4f}")

Epoch 01/7 | Train Loss: 3.8895 | Val Loss: 2.1768 | Val AUC: 0.9472 | LR: 2.85e-04
  -> Saved! Best AUC: 0.9472


Epoch 02/7 | Train Loss: 2.7381 | Val Loss: 1.6759 | Val AUC: 0.9640 | LR: 2.44e-04
  -> Saved! Best AUC: 0.9640


Epoch 03/7 | Train Loss: 2.4232 | Val Loss: 1.4870 | Val AUC: 0.9686 | LR: 1.84e-04
  -> Saved! Best AUC: 0.9686


Epoch 04/7 | Train Loss: 2.1751 | Val Loss: 1.3461 | Val AUC: 0.9750 | LR: 1.17e-04
  -> Saved! Best AUC: 0.9750


Epoch 05/7 | Train Loss: 2.0901 | Val Loss: 1.2857 | Val AUC: 0.9773 | LR: 5.73e-05
  -> Saved! Best AUC: 0.9773


Epoch 06/7 | Train Loss: 1.9509 | Val Loss: 1.2206 | Val AUC: 0.9809 | LR: 1.58e-05
  -> Saved! Best AUC: 0.9809


Epoch 07/7 | Train Loss: 1.8808 | Val Loss: 1.2106 | Val AUC: 0.9813 | LR: 1.00e-06
  -> Saved! Best AUC: 0.9813

Best Val AUC: 0.9813


In [26]:
from google.colab import drive
import joblib, os

drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/BirdCLEF_Project'
os.makedirs(save_path, exist_ok=True)

!cp best_efficientnet_b0.pth {save_path}/
joblib.dump(le, f"{save_path}/label_encoder.pkl")
pd.DataFrame(history).to_csv(f"{save_path}/efficientnet_history.csv", index=False)
print(f"Saved to {save_path}")

Mounted at /content/drive
Saved to /content/drive/MyDrive/BirdCLEF_Project


In [28]:
print("=" * 50)
print(f"{'Model':<28} {'Metric':<18} {'Score'}")
print("=" * 50)
print(f"{'ResNet18 — Participant 1':<28} {'Val Accuracy':<18} 0.4520")
print(f"{'EfficientNet-B0 — Participant 2':<28} {'Macro ROC-AUC':<18} {best_auc:.4f}")
print("=" * 50)

Model                        Metric             Score
ResNet18 — Participant 1     Val Accuracy       0.4520
EfficientNet-B0 — Participant 2 Macro ROC-AUC      0.9813


# Model 2 — EfficientNet-B0

## Architecture

`timm.create_model('efficientnet_b0', pretrained=True)` with `drop_rate=0.3`. Final FC layer → 234 classes.

## Mel Spectrogram

| Parameter | Value |
|---|---|
| Sample rate | 32,000 Hz |
| Duration | 5s |
| n_fft / hop / n_mels | 1024 / 512 / 128 |
| f_min / f_max | 50 / 14,000 Hz |
| Normalization | (dB + 80) / 80 |
| Image size | 128 × 256 |

## Augmentations

- **Waveform:** Gaussian noise (p=0.3)
- **Spectrogram:** FrequencyMasking(15) + TimeMasking(30)
- **Batch:** Mixup alpha=0.4 (p=0.5)

## Training

| Parameter | Value |
|---|---|
| Loss | Focal Loss (γ=2.0) + label smoothing 0.05 |
| Optimizer | AdamW lr=3e-4, wd=1e-4 |
| Scheduler | CosineAnnealingLR → 1e-6 |
| Epochs | 15 |
| Batch size | 32 |

Validation: fold 0 from Participant 1's StratifiedKFold split. Metric: **Macro-averaged ROC-AUC**.

## Comparison with Previous Competitions

| Year | Location | Species | Metric | Key Technique |
|---|---|---|---|---|
| 2021 | Africa | 397 | cmAP | Pretrained on prior BirdCLEF data |
| 2022 | East Africa | 152 | cmAP | No-call class + pseudo-labels |
| 2023 | East Africa | 264 | padded cmAP | SED attention head |
| 2024 | Borneo | 182 | padded cmAP | Iterative pseudo-labeling |
| 2025 | Colombia | 206 | padded cmAP | Model Soup + GeM + ensemble |
| **2026** | **Pantanal** | **234** | **Macro ROC-AUC** | TBD |

Our EfficientNet-B0 corresponds to ~2022–2023 level: standard mel pipeline + pretrained CNN + Mixup/SpecAugment. Next steps: SED head, pseudo-labeling, ensemble.
